# A2 Qwen3-1.7B — full 2 epochs on Kaggle
Attach the ZIP bundle as a **private** dataset. Enable Internet and GPU.
Same 1054 training samples, 84 validation, 56 development challenge; no reserved test.
Start again from the original RAFT v2 adapter, not checkpoint-75 of sentence v3.
Save both complete epochs. Evaluate both on validation and challenge; no loss-based reload.
Use one GPU even when T4 x2 is allocated. Results and resumable checkpoints: /kaggle/working.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, os, subprocess, sys
matches = list(Path('/kaggle/input').rglob('a2_sentence_v3_kaggle_bundle.bundle'))
if not matches:
    matches = list(Path('/kaggle/input').rglob('a2_sentence_v3_kaggle_bundle.zip'))
if len(matches) != 1:
    raise RuntimeError('Attach the private dataset containing a2_sentence_v3_kaggle_bundle.zip first.')
WORK = Path('/kaggle/working/a2_code')
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(matches[0]) as z:
    manifest = json.loads(z.read('bundle_manifest.json'))
    assert set(z.namelist()) == set(manifest['files_sha256']) | {'bundle_manifest.json'}
    for name, digest in manifest['files_sha256'].items():
        assert not PurePosixPath(name).is_absolute() and '..' not in PurePosixPath(name).parts
        assert hashlib.sha256(z.read(name)).hexdigest() == digest, name
    z.extractall(WORK)
print('Verified bundle:', matches[0])


In [ ]:
%pip install -q -r /kaggle/working/a2_code/requirements-colab-a2.txt

In [ ]:
RUN_DIR = Path('/kaggle/working/a2_sentence_v3_kaggle_01')
env = dict(os.environ, PYTHONPATH=str(WORK), HF_HOME='/tmp/a2_hf_cache',
           CUDA_VISIBLE_DEVICES='0', TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')
def run_stage(stage):
    command = [sys.executable, '-u', '-m', 'scripts.train_a2_sentence_kaggle',
               '--stage', stage, '--output-dir', str(RUN_DIR)]
    with subprocess.Popen(command, cwd=WORK, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout: print(line, end='', flush=True)
        if process.wait(): raise RuntimeError('Failed stage: '+stage)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# One run; do not launch a second GPU session in parallel.
for stage in ['preflight', 'train',
              'epoch1_after', 'challenge_epoch1_after', 'after', 'challenge_after', 'export']:
    print('\nSTAGE:', stage, flush=True)
    run_stage(stage)


In [ ]:
summary = {}
for name in ['epoch1_after','after','challenge_epoch1_after','challenge_after']:
    summary[name] = json.loads((RUN_DIR/name/'metrics.json').read_text())
(RUN_DIR/'checkpoint_comparison.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
figures = RUN_DIR/'figures'
figures.mkdir(exist_ok=True)
logs = json.loads((RUN_DIR/'training_log.json').read_text())
fig, ax = plt.subplots()
for key, label in [('loss','Train loss'),('eval_loss','Validation loss')]:
    points = [(r['step'],r[key]) for r in logs if key in r]
    if points: ax.plot(*zip(*points),marker='o',label=label)
ax.set(xlabel='Optimizer step',ylabel='Loss'); ax.legend()
fig.tight_layout(); fig.savefig(figures/'loss.png',dpi=180); plt.close(fig)
for stage in ['epoch1_after','after']:
    m = summary[stage]
    matrix = m['confusion_matrix']
    if isinstance(matrix,dict):
        raise ValueError('Unexpected confusion matrix format; retain metrics for inspection')
    fig, ax = plt.subplots()
    ax.imshow(matrix,cmap='Blues')
    for i,row in enumerate(matrix):
        for j,value in enumerate(row): ax.text(j,i,str(value),ha='center',va='center')
    ax.set(xticks=range(3),xticklabels=['draft','insufficient','invalid'],yticks=range(2),
           yticklabels=['draft','insufficient'],xlabel='Predicted',ylabel='Expected',title=stage)
    fig.tight_layout(); fig.savefig(figures/(stage+'_04_confusion_matrix.png'),dpi=180); plt.close(fig)
    labels = ['Status macro-F1','Sentence micro-F1','Sentence F1 answerable']
    values = [m['status_macro_f1'],m['sentence_micro_f1'],m['selection_f1_on_answerable']]
    fig, ax = plt.subplots(figsize=(8,4))
    bars = ax.bar(labels,values); ax.bar_label(bars,fmt='%.3f')
    ax.set(ylim=(0,1.15),title=stage+' (reference matching, not semantics)')
    fig.tight_layout(); fig.savefig(figures/(stage+'_f1.png'),dpi=180); plt.close(fig)
    fig, ax = plt.subplots()
    decisions = ['draft','insufficient_evidence']
    bars = ax.bar(decisions,[m['per_status'][d]['f1'] for d in decisions])
    ax.bar_label(bars,fmt='%.3f'); ax.set(ylim=(0,1.15),ylabel='F1',title=stage)
    fig.tight_layout(); fig.savefig(figures/(stage+'_05_f1_by_decision.png'),dpi=180); plt.close(fig)
run_stage('export')
from IPython.display import FileLink, display
display(FileLink(str(RUN_DIR.parent/(RUN_DIR.name+'_results.zip'))))
print('Epoch 1 adapter:', RUN_DIR/'epoch_adapters/epoch_1')
print('Epoch 2 adapter:', RUN_DIR/'adapter')
print('PNG charts:', RUN_DIR/'figures')
